In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
order_geo = pd.read_csv("../data/raw/order_geolocation.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")


# match the product_id and their product category name

In [ ]:
display(products.head(2))


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,Art,44.0,276.0,1.0,1000.0,30.0,18.0,20.0


In [ ]:
order_product = order_geo[['order_id', 'product_id', 'customer_state', 'seller_state']].merge(products[['product_id', 'product_category_name']], how='left', on='product_id')
order_product.head()

,order_id,product_id,customer_state,seller_state,product_category_name
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,RJ,SP,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,SP,SP,pet Shop
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,MG,MG,furniture_decoration
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,SP,SP,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,SP,PR,garden_tools


In [ ]:
order_product.info() # show some order's products have no category, should be deleted

<class 'pandas.core.frame.DataFrame'>
Int64Index: 98666 entries, 0 to 98665
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   order_id               98666 non-null  object
 1   product_id             98666 non-null  object
 2   customer_state         98666 non-null  object
 3   seller_state           98666 non-null  object
 4   product_category_name  97250 non-null  object
dtypes: object(5)
memory usage: 4.5+ MB


In [ ]:
order_product = order_product.dropna(axis=0, how = 'any')
order_product.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 97250 entries, 0 to 98665
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   order_id               97250 non-null  object
 1   product_id             97250 non-null  object
 2   customer_state         97250 non-null  object
 3   seller_state           97250 non-null  object
 4   product_category_name  97250 non-null  object
dtypes: object(5)
memory usage: 4.5+ MB


In [ ]:
state_product_dm = order_product.groupby(['product_category_name', 'customer_state']).size().reset_index(name='demand_freq')
# find out the top 3 products demand
top_dm = state_product_dm.groupby('customer_state').apply(lambda x: x.nlargest(3, 'demand_freq')).reset_index(drop=True)
display(state_product_dm)

In [ ]:
state_product_sp = order_product.groupby(['product_category_name', 'seller_state']).size().reset_index(name='supply_freq')
# find out the top 3 products supply
top_sp = state_product_sp.groupby('seller_state').apply(lambda x: x.nlargest(3, 'supply_freq')).reset_index(drop=True)
display(state_product_dm)

In [ ]:
prodcut_trade = pd.merge(state_product_dm, state_product_sp, how='outer', left_on=['customer_state', 'product_category_name'], right_on=['seller_state', 'product_category_name'])

In [ ]:
prodcut_trade # show some columns have null value when mergging

,product_category_name,customer_state,demand_freq,seller_state,supply_freq
0,Art,AM,1.0,NaN,NaN
1,Art,AP,2.0,NaN,NaN
2,Art,BA,10.0,NaN,NaN
3,Art,DF,5.0,DF,1.0
4,Art,ES,1.0,NaN,NaN
...,...,...,...,...,...
1373,furniture_room,NaN,NaN,CE,4.0
1374,insurance_and_services,NaN,NaN,SP,2.0
1375,landline_telephony,NaN,NaN,RN,1.0
1376,music,NaN,NaN,SC,2.0


In [ ]:
# fill in the null value in supply & demand freq
prodcut_trade.supply_freq.fillna(0, inplace=True)
prodcut_trade.demand_freq.fillna(0, inplace=True)

In [ ]:
# fill in the null value in customer_state & seller_state
prodcut_trade['customer_state'].fillna(prodcut_trade['seller_state'], inplace=True)

# drop the duplicated column seller_state
prodcut_trade.drop('seller_state', axis=1, inplace=True)

In [ ]:
# add a column to show the differences between purchase and sales of the same states
prodcut_trade['sale_gap'] = prodcut_trade.demand_freq - prodcut_trade.supply_freq


In [ ]:
# make the differences be absolute values but seems no need to do
prodcut_trade['sale_gap_abs'] = prodcut_trade.sale_gap.abs()
prodcut_trade

,product_category_name,state,demand_freq,supply_freq,sale_gap,sale_gap_abs
0,Art,AM,1,0,1,1
1,Art,AP,2,0,2,2
2,Art,BA,10,0,10,10
3,Art,DF,5,1,4,4
4,Art,ES,1,0,1,1
...,...,...,...,...,...,...
1373,furniture_room,CE,0,4,-4,4
1374,insurance_and_services,SP,0,2,-2,2
1375,landline_telephony,RN,0,1,-1,1
1376,music,SC,0,2,-2,2


In [ ]:
# all float value change to int
prodcut_trade[['demand_freq', 'supply_freq', 'sale_gap', 'sale_gap_abs']] = prodcut_trade[['demand_freq', 'supply_freq', 'sale_gap', 'sale_gap_abs']].round(0).astype(int)
prodcut_trade

,product_category_name,state,demand_freq,supply_freq,sale_gap,sale_gap_abs
0,Art,AM,1,0,1,1
1,Art,AP,2,0,2,2
2,Art,BA,10,0,10,10
3,Art,DF,5,1,4,4
4,Art,ES,1,0,1,1
...,...,...,...,...,...,...
1373,furniture_room,CE,0,4,-4,4
1374,insurance_and_services,SP,0,2,-2,2
1375,landline_telephony,RN,0,1,-1,1
1376,music,SC,0,2,-2,2


In [ ]:
# rename the customer_state to state
prodcut_trade.rename(columns={'customer_state':'state'}, inplace=True)

In [ ]:
prodcut_trade_done = prodcut_trade.copy()
prodcut_trade_done

In [ ]:
# export file of all sales gap
prodcut_trade_done.to_csv('../data/raw/recommend_sales_gap.csv', index=False)

In [ ]:
# import file of all sales gap
prodcut_trade_done = pd.read_csv("../data/raw/recommend_sales_gap.csv")

In [ ]:
# find out the top x sale gap, which will be the products' recommendation of each state. Olist should consider add more sellers of the products
recom_prod = prodcut_trade_done.groupby('state').apply(lambda x: x.nlargest(5, 'sale_gap')).reset_index(drop=True)
recom_prod

In [ ]:
# export file of top 3 sales gap
recom_prod.to_csv('../data/processed/recommend_sales_gap_5.csv', index=False)

In [ ]:
# done